In [1]:
from src.graph.orchastrator import HierarchicalAgent
from src.graph.orchastrator.preprocessing import prepare_orchestrator_input
from src.database.data_preprocessor import preprocess_user

/Users/rogier/dev/joho/advies_agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Preprocess a user by aanvraag_id
user = preprocess_user(92)
inputs = prepare_orchestrator_input(user)

print(f"Providers: {inputs['available_providers']}")
print(f"Product descriptions loaded: {list(inputs['product_descriptions'].keys())}")
print(f"User profile keys: {list(inputs['user_profile'].keys())}")

Providers: ['oom_wib', 'allianz_globetrotter', 'goudse_expat_pakket', 'oom_tib', 'goudse_ngo_zendelingen', 'special_isis', 'globality_yougenio', 'expatriate_group', 'International Expat Insurance', 'goudse_working_nomad']
Product descriptions loaded: ['oom_wib', 'allianz_globetrotter', 'goudse_expat_pakket', 'oom_tib', 'goudse_ngo_zendelingen', 'special_isis', 'globality_yougenio', 'expatriate_group', 'goudse_working_nomad']
User profile keys: ['aanvraag_id', 'family', 'deductible_requested', 'hoofdreden_verblijf', 'toelichting_hoofdreden', 'verwachte_duur_verblijf', 'toelichting_duur', 'interesse_internationale_aov', 'interesse_zkv', 'zkv_dekkingsvariant', 'zkv_eigen_risico_voorkeur', 'zkv_periode', 'huidige_verzekeraar', 'voorkeur_verzekeraar', 'medische_bijzonderheden', 'specifieke_wensen_zkv', 'dekking_zwangerschap', 'sporten_activiteiten', 'sport_semiprofessioneel', 'eerder_contact_joho', 'advies_vorm']


In [3]:
user

PreprocessedUser(aanvraag_id=92, email=None, premiums={'oom_wib': {'Comfort': {'total': 13793.28, 'deductible': 500.0, 'per_person': {'main': 6896.64, 'partner': 6896.64}}, 'Deluxe': {'total': 23971.920000000002, 'deductible': 500.0, 'per_person': {'main': 11985.960000000001, 'partner': 11985.960000000001}}, 'Regular': {'total': 9594.72, 'deductible': 500.0, 'per_person': {'main': 4797.36, 'partner': 4797.36}}}, 'allianz_globetrotter': {'Standaard - All-In pakket': {'total': 1108.8000000000002, 'deductible': 100.0, 'per_person': {'main': 554.4000000000001, 'partner': 554.4000000000001}}, 'Standaard - Extra pakket': {'total': 606.0, 'deductible': 100.0, 'per_person': {'main': 303.0, 'partner': 303.0}}, 'Super - All-In pakket': {'total': 1378.8000000000002, 'deductible': 100.0, 'per_person': {'main': 689.4000000000001, 'partner': 689.4000000000001}}, 'Super - Extra pakket': {'total': 876.0, 'deductible': 100.0, 'per_person': {'main': 438.0, 'partner': 438.0}}}, 'goudse_expat_pakket': {'E

In [4]:
inputs

{'user_profile': {'aanvraag_id': 92,
  'family': [{'role': 'main', 'age': 28, 'birth_date': '1998-01-17'},
   {'role': 'partner', 'age': 25, 'birth_date': '2000-06-30'}],
  'deductible_requested': 500.0,
  'hoofdreden_verblijf': 'Werken voor eigen onderneming, Werken & reizen, Emigratie',
  'toelichting_hoofdreden': 'Geemigreerd naar UAE, reizend door Europa',
  'verwachte_duur_verblijf': '2 - 5 jaar',
  'toelichting_duur': 'Geen',
  'interesse_internationale_aov': 'Nee',
  'interesse_zkv': 'Ja',
  'zkv_dekkingsvariant': 'Gemiddelde dekking (Medium)',
  'zkv_eigen_risico_voorkeur': '€ 500',
  'zkv_periode': 'Voor meerdere jaren',
  'huidige_verzekeraar': 'Zorgzaam',
  'voorkeur_verzekeraar': 'Nee',
  'medische_bijzonderheden': 'Nee',
  'specifieke_wensen_zkv': 'Nee',
  'dekking_zwangerschap': 'Nee',
  'sporten_activiteiten': 'Surfen, sportschool ',
  'sport_semiprofessioneel': 'Nee',
  'eerder_contact_joho': 'Nee',
  'advies_vorm': 'Kort'},
 'premiums': {'oom_wib': {'Comfort': {'total'

In [5]:
agent = HierarchicalAgent()
thread_id = "my-run-3"

In [6]:
# 1. Maak een variabele aan om de totale state in op te bouwen
final_result = {}
stream_error = None

# 2. Stream execution — see each node as it completes
try:
    for event in agent.stream(**inputs, thread_id=thread_id):
        node_name = list(event.keys())[0]
        updates = event[node_name]
        
        # 3. Voeg de data van deze node toe aan onze verzamel-variabele
        final_result.update(updates)

        print(f"\n{'='*60}")
        print(f"Node: {node_name}")
        print(f"Returned keys: {list(updates.keys())}")

        # Show key info per node
        if node_name == "user_agent":
            pc = updates.get("parsed_constraints", {})
            aspects = pc.get("retrieval_aspects", [])
            print(f"  Retrieval aspects: {[a['aspect'] for a in aspects]}")
            print(f"  Interpreted needs: {pc.get('interpreted_needs', [])}")

        elif node_name == "build_tracker":
            tracker = updates.get("retrieval_tracker", {})
            for p, data in tracker.items():
                levels = list(data.get("coverage_levels", {}).keys())
                print(f"  {p}: {len(data['aspects'])} aspects, levels={levels}")

        elif node_name == "orchestrator_assess":
            tasks = updates.get("retrieval_tasks", [])
            tracker = updates.get("retrieval_tracker", {})
            dropped = [p for p, d in tracker.items() if d.get("status") == "dropped"]
            print(f"  Tasks dispatched: {len(tasks)}")
            for t in tasks:
                print(f"    -> {t['provider']}: {t['aspect']} | {t['query'][:80]}")
            if dropped:
                print(f"  Dropped providers: {dropped}")

        elif node_name == "retriever":
            summaries = updates.get("retrieval_summaries", [])
            for s in summaries:
                if s:
                    print(f"  {s.get('provider')}/{s.get('aspect')}: confidence={s.get('confidence')}")
                    print(f"    {s.get('overall_summary', '')[:120]}...")

        elif node_name == "merge_results":
            print(f"  Iteration: {updates.get('retrieval_iteration')}")

        elif node_name == "evaluator_step1":
            qa = updates.get("qualitative_assessment", {})
            assessments = qa.get("assessments", [])
            for a in assessments:
                print(f"  {a['provider']} {a['coverage_level']}: €{a['premium']}")
                print(f"    Fit: {a['overall_fit'][:100]}...")

        elif node_name == "evaluator_step2":
            rec = updates.get("recommendation", {})
            print("  TOP RECOMMENDATIONS:")
            for r in rec.get("top_recommendations", []):
                print(f"    {r['provider']} {r['coverage_level']} (€{r['premium']})")
            print("  BUDGET RECOMMENDATIONS:")
            for r in rec.get("budget_recommendations", []):
                print(f"    {r['provider']} {r['coverage_level']} (€{r['premium']})")

except Exception as e:
    stream_error = e
    print(f"\n{'!'*60}")
    print(f"ERROR at node stream: {type(e).__name__}: {e}")
    print(f"Partial results saved in 'final_result' with keys: {list(final_result.keys())}")

# 4. Nadat de loop klaar is, heb je nu de complete state tot je beschikking!
print(f"\n{'*'*60}")
print("STREAM COMPLEET." if stream_error is None else "STREAM INCOMPLETE (partial results saved).")
print("Alle keys in 'final_result':", list(final_result.keys()))


Node: user_agent
Returned keys: ['parsed_constraints']
  Retrieval aspects: ['Geographic scope of coverage (UAE residence and travel in Europe) and definition of home country/host country', 'Outpatient and inpatient benefits limits under the Medium plan (annual maximums, sublimits, hospital room type)', 'Deductible/excess application (€500) and how it applies (per year vs per claim; inpatient vs outpatient; family deductible)', 'Emergency care, medical evacuation, repatriation, and assistance services', 'Sports and hazardous activities coverage (surfing, gym) and exclusions', 'Coverage during work for self-employed persons and any occupational exclusions', 'Pre-existing conditions definition, underwriting, and waiting periods', 'Maternity and pregnancy benefits exclusions (and ability to add later)', 'Policy term, renewal conditions for multi-year coverage, and cancellation rules while abroad', 'Direct billing/network providers in UAE and claims process for care in Europe', "Existing 

In [9]:
final_result["parsed_constraints"] = final_result.get("parsed_constraints", {})
final_result["retrieval_tracker"] = final_result.get("retrieval_tracker", {})
final_result["retrieval_tasks"] = final_result.get("retrieval_tasks", [])
final_result["retrieval_summaries"] = final_result.get("retrieval_summaries", [])
final_result["qualitative_assessment"] = final_result.get("qualitative_assessment", {})
final_result["recommendation"] = final_result.get("recommendation", {})

In [11]:
final_result["recommendation"] 

{'top_recommendations': [{'provider': 'goudse_expat_pakket',
   'coverage_level': 'Optimaal',
   'premium': 2840.0,
   'reasoning': 'Voor jullie profiel (28/25 jaar, zelfstandig ondernemer, woonachtig in de VAE en regelmatig reizen door Europa, looptijd 2–5 jaar) is dit pakket inhoudelijk het meest ‘expat-proof’. De regio-opzet sluit logisch aan: staat de VAE als woonland op de polis, dan valt medische dekking in een regio waarin de VAE expliciet is opgenomen (Region C) en Europa is altijd inbegrepen (Region A). Het gekozen eigen risico van €500 is bovendien een gezins-eigen-risico per verzekeringsjaar, wat bij twee personen vaak gunstiger uitpakt dan €500 p.p. De SOS/assistance is sterk ingericht (ANWB alarmcentrale, expliciete evacuatie- en repatriëringscomponent), wat in de VAE context (hoge kosten/complexe zorgketen) extra relevant is.',
   'trade_offs': 'Je kiest hier voor een expatpakket met goede geografische logica en sterke hulpverlening, maar accepteert dat sportdekking rond 

In [12]:
final_result["parsed_constraints"]

{'core': {'age': 28,
  'destination': 'Verenigde Arabische Emiraten (woonland) en reizen door Europa',
  'duration': '2 - 5 jaar (meerdere jaren)',
  'employment_type': 'Zelfstandig ondernemer (eigen onderneming)',
  'trip_type': 'Expat/Emigratie met doorlopend reizen (long-stay/multi-country)'},
 'interpreted_needs': ['Internationale ziektekostenverzekering (ZKV) voor langdurig verblijf (2–5 jaar) met dekking passend bij emigratie/expat-status in de UAE, inclusief zorg buiten het woonland tijdens reizen door Europa.',
  'Medium dekking met eigen risico €500: behoefte aan balans tussen premie en dekking; verwacht gebruik is ‘normaal’ (geen medische bijzonderheden) maar wel behoefte aan solide basiszorg en onverwachte zorgkosten.',
  'Wereldwijde dekking/territorialiteit: omdat ze wonen in de UAE en regelmatig in Europa reizen, moet de polis zorg in meerdere landen dekken (woonland + reislanden), incl. spoed en planbare zorg.',
  'Dekking voor sporten: surfen en sportschool (recreatief)

In [16]:
final_result["retrieval_tracker"]


{'oom_wib': {'status': 'active',
  'aspects': {'Geographic scope of coverage (UAE residence and travel in Europe) and definition of home country/host country': {'status': 'retrieved',
    'summaries': [{'provider': 'oom_wib',
      'aspect': 'Geographic scope of coverage (UAE residence and travel in Europe) and definition of home country/host country',
      'overall_summary': '**Waar is de verzekering geldig (territorial scope / area of cover):** De geldigheid hangt af van de gekozen **regio (Region)**. De documenten noemen 5 regio’s met steeds ‘worldwide’ dekking, maar met uitsluitingen per regio: **Region A (World)**: ‘no exclusions; you are insured worldwide’. **Region B+ (World except region A)**: wereldwijd, maar **niet verzekerd in the United States**. **Region B (World except region A, B+)**: wereldwijd, maar **niet verzekerd in the United States and Singapore**. **Region C+ (World except region A, B+, B)**: wereldwijd, maar **niet verzekerd in the United States, Singapore, Chi

In [17]:
final_result["retrieval_tasks"] 

[{'provider': 'goudse_expat_pakket',
  'query': 'outpatient inpatient benefits annual maximum sublimits hospital room type Standard Optimal Excellent coverage levels',
  'aspect': 'Outpatient and inpatient benefits limits under the Medium plan (annual maximums, sublimits, hospital room type)'},
 {'provider': 'goudse_expat_pakket',
  'query': 'pre-existing conditions definition underwriting waiting periods moratorium',
  'aspect': 'Pre-existing conditions definition, underwriting, and waiting periods'},
 {'provider': 'goudse_expat_pakket',
  'query': 'policy term renewal multi-year cancellation rules termination while abroad',
  'aspect': 'Policy term, renewal conditions for multi-year coverage, and cancellation rules while abroad'},
 {'provider': 'globality_yougenio',
  'query': 'outpatient inpatient benefits annual maximum sublimits hospital room type Essential Classic coverage levels',
  'aspect': 'Outpatient and inpatient benefits limits under the Medium plan (annual maximums, subli

In [18]:
final_result["retrieval_summaries"] 

[]

In [19]:
final_result["qualitative_assessment"]


{'assessments': [{'provider': 'oom_wib',
   'coverage_level': 'Regular',
   'premium': 9594.72,
   'overall_fit': 'Past qua looptijd (doorlopend) en opzet goed bij een 2–5 jaar expat-/emigratiesituatie, maar de regionale keuze is hier cruciaal omdat UAE in een van de regio’s expliciet is uitgesloten. Met €500 eigen risico per persoon per jaar sluit het aan bij de voorkeur, maar er is nog te weinig zekerheid over sportdekking (surfen) en exacte medische maxima/sublimieten.',
   'strengths': ['Sterke expat-propositie: bedoeld voor wonen/werken/reizen in het buitenland met modulaire opzet en meerjarige/doorlopende duur.',
    'Eigen risico werkt helder: €500 is per persoon per verzekeringsjaar en bij doorlopende ziekenhuisopname over jaargrens slechts één keer toegepast.',
    'SOS-dekking geldt volgens de voorwaarden wereldwijd en kent geen eigen risico, wat bij evacuatie/repatriëring relevant is.'],
   'weaknesses': ['Territorialiteit kan een showstopper zijn: in Region C is UAE explici

In [20]:
final_result["retrieval_tracker"] 

{'oom_wib': {'status': 'active',
  'aspects': {'Geographic scope of coverage (UAE residence and travel in Europe) and definition of home country/host country': {'status': 'retrieved',
    'summaries': [{'provider': 'oom_wib',
      'aspect': 'Geographic scope of coverage (UAE residence and travel in Europe) and definition of home country/host country',
      'overall_summary': '**Waar is de verzekering geldig (territorial scope / area of cover):** De geldigheid hangt af van de gekozen **regio (Region)**. De documenten noemen 5 regio’s met steeds ‘worldwide’ dekking, maar met uitsluitingen per regio: **Region A (World)**: ‘no exclusions; you are insured worldwide’. **Region B+ (World except region A)**: wereldwijd, maar **niet verzekerd in the United States**. **Region B (World except region A, B+)**: wereldwijd, maar **niet verzekerd in the United States and Singapore**. **Region C+ (World except region A, B+, B)**: wereldwijd, maar **niet verzekerd in the United States, Singapore, Chi